## Connecting to MySQL with `mysql-connector-python`


In [1]:
import mysql.connector as connector

Create a connection pool and borrow a connection from it. This is more
efficient than opening a brand new connection for every query.


In [13]:
import os
from pathlib import Path

from mysql.connector.pooling import MySQLConnectionPool
from mysql.connector import Error
import mysql.connector as connector

# Credentials are read from a local, git-ignored .env file.
# Copy .env.example to .env and fill in your own values.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    # Minimal fallback so the notebook also runs without python-dotenv.
    env_path = Path.cwd().parent / ".env"
    if not env_path.exists():
        env_path = Path.cwd() / ".env"
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

dbconfig = {
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("DB_PORT", "3306")),
    "database": os.getenv("DB_NAME", "LittleLemonDB"),
    "auth_plugin": "mysql_native_password",
}

if not dbconfig["user"] or not dbconfig["password"]:
    raise RuntimeError(
        "Missing DB credentials. Copy .env.example to .env and fill it in."
    )

try:
    pool = MySQLConnectionPool(pool_name="ll_pool_a",
                               pool_size=2,  # default is 5
                               **dbconfig)
    print("The connection pool is created with a name: ", pool.pool_name)
    print("The pool size is:", pool.pool_size)

except Error as er:
    print("Error code:", er.errno)
    print("Error message:", er.msg)

try:
    print("Getting a connection from the pool....")
    connection1 = pool.get_connection()
    print("connection1 connected")
except Exception:
    print("No more connections are available.")
    print("Adding a new connection to the pool.")

    # Create a connection and add it to the pool
    connection = connector.connect(**dbconfig)
    pool.add_connection(cnx=connection)
    print("A new connection has been added to the pool.\n")

    print("Getting a connection from the pool.")
    connection1 = pool.get_connection()
    print("connection1 connected after adding a new connection to the pool")

print("Creating a cursor object...")

cursor = connection1.cursor()

print("cursor object created")


The connection pool is created with a name:  ll_pool_a
The pool size is: 2
Getting a connection from the pool....
connection1 connected
Creating a cursor object...
cursor object created


### First query: list the tables in the database


In [14]:
show_tables_query = "SHOW tables" 
cursor.execute(show_tables_query)

### Printing the query results


In [15]:
result = cursor.fetchall()

print("Tables in LittleLemonDB are :")

for (database,) in result:
    print(database.decode('utf-8'))

Tables in LittleLemonDB are :
Virtual_OrdersView
bookings
customers
delivery_status
menu
menu_item
orders
staff


### Reporting: customers who spent more than $60

Return the full name, contact details and bill amount for every customer whose
order total was greater than $60, for a promotional campaign.


In [20]:
query = """SELECT
    full_name       AS 'Customer Name',
    contact_numbers AS 'Contact Numbers',
    email           AS Email,
    total_cost      AS 'Bill Amount'
FROM customers AS cst
RIGHT JOIN orders AS odr ON cst.customer_id = odr.customer_id
WHERE total_cost > 60"""

cursor.execute(query)

cols = cursor.column_names
print("All the customers with a bill greater than $60\n")
for row in cursor.fetchall():
    print("\t\t\t", cols[0], " : ", row[0])
    print("\t\t\t", cols[1], " : ", row[1])
    print("\t\t\t", cols[2], " : ", row[2])
    print("\t\t\t", cols[3], " : ", row[3])


All the Customers with Bill More than $60

			 Customer Name  :  Andrew Martin
			 Contact Numbers  :  001-366-840-2500
			 Email  :  yvasquez@example.net
			 Bill Amount  :  3059.84

			 Customer Name  :  Jimmy Williams
			 Contact Numbers  :  001-228-713-4510x2832
			 Email  :  stephanie35@example.net
			 Bill Amount  :  2041.79

			 Customer Name  :  Natalie Waller
			 Contact Numbers  :  243.699.8943x34842
			 Email  :  george02@example.org
			 Bill Amount  :  4891.28

			 Customer Name  :  Stephanie Diaz
			 Contact Numbers  :  9534755744
			 Email  :  amanda57@example.com
			 Bill Amount  :  1807.56

			 Customer Name  :  Andrew Nguyen
			 Contact Numbers  :  208-637-7453x999
			 Email  :  karenward@example.org
			 Bill Amount  :  2067.26

			 Customer Name  :  Carol Copeland
			 Contact Numbers  :  (620)763-6545x10781
			 Email  :  michellepeterson@example.org
			 Bill Amount  :  3794.72

			 Customer Name  :  Kimberly Kirby
			 Contact Numbers  :  +1-850-259-2704x0766
			 Email